# Mistral

**Mistral AI** is a Paris-based lab that ships both **open-weight** models you download and run yourself (Mistral 7B, Mixtral, Mistral Small/Nemo, Codestral, …) **and** a hosted **API** ("La Plateforme") with closed premier models (Mistral Large, Pixtral, …). Its calling cards: **efficiency** (sliding-window attention, grouped-query attention, mixture-of-experts) and **permissive Apache-2.0 licensing** on much of the open line.

**Domain:** Proprietary Models & Coding AI  ·  **from study list**  ·  **runnable:** yes  ·  _HF weights (Apache-2.0) / hosted API (live cells gate on `os.getenv`)_

## 1. What & Why

**Mistral AI** is a European (French) frontier lab founded in 2023 by alumni of Meta and DeepMind. It occupies an unusual middle ground in this domain: it gives you **both** worlds.

- **Open-weight models** — you download the weights from Hugging Face and run them yourself, like LLaMA/Qwen. Much of this line is **Apache 2.0** (genuinely permissive, OSI-style — no MAU clause).
- **A hosted API** — *La Plateforme* — for the models that aren't open (Mistral Large, the multimodal Pixtral Large, etc.), plus **Le Chat** (the consumer chat app).

**The roster you actually meet today:**

- **Mistral 7B** — the model that put them on the map: a dense 7B that beat much larger models at release, thanks to **sliding-window attention** + **grouped-query attention**. Apache 2.0.
- **Mixtral 8x7B / 8x22B** — **sparse mixture-of-experts (MoE)**: ~47B / ~141B *total* parameters but only ~13B / ~39B *active* per token, so you get big-model quality at small-model inference cost. Apache 2.0.
- **Mistral Small 3 / 3.1 (24B)** and **Mistral Nemo (12B)** — current efficient open workhorses (Nemo was co-built with NVIDIA; 128K context). Apache 2.0.
- **Codestral (22B)** — code-specialized; strong fill-in-the-middle (FIM) for IDE autocomplete. Non-production weights under the MNPL license; available via API for commercial use.
- **Mistral Large 2 / Pixtral** — the **closed, premier** API-only models (Pixtral adds vision). Top quality, API-metered.

**The problem it solves / why reach for it:**

- **Efficiency per dollar.** The whole brand is "more quality per FLOP." MoE (Mixtral) and sliding-window attention mean you pay for ~13B of compute and get ~47B-class answers.
- **Open weights with a *real* open license.** Apache 2.0 on Mistral 7B / Mixtral / Nemo / Small means self-hosting, fine-tuning, and commercial use with **no usage-cap clause** (unlike Llama's Community License). For many teams that's the deciding factor.
- **One vendor, two deployment modes.** Prototype against the API, then self-host the open model with the *same* prompt format — or mix (open model for bulk, Mistral Large for the hard 5%).
- **European data residency.** EU-headquartered with EU hosting options matters for GDPR-sensitive workloads.

**When NOT to:** if you want the **absolute frontier** on hard reasoning/coding, the top closed models (Claude, GPT, Gemini) still tend to edge out Mistral Large. If you want zero ops and low volume, any hosted API beats babysitting GPUs. And note the open/closed split: **Codestral and Mistral Large are *not* Apache 2.0** — don't assume every Mistral model is free to self-host commercially.

## 2. Mental Model

Think of Mistral as **"an open-weight family *and* an OpenAI-compatible API from the same lab — joined by one efficiency obsession."**

```
                         Mistral AI (Paris)
                                 │
            ┌────────────────────┴─────────────────────┐
            ▼                                           ▼
   OPEN WEIGHTS (Apache-2.0)                  HOSTED API — La Plateforme
   download from HF, run yourself             api.mistral.ai (OpenAI-ish)
   ┌──────────────────────────┐               ┌──────────────────────────┐
   │ Mistral 7B   (dense)     │               │ Mistral Large 2 (closed) │
   │ Mixtral 8x7B / 8x22B (MoE)│              │ Pixtral Large  (vision)  │
   │ Mistral Nemo 12B, Small 24B│             │ Codestral (code, FIM)    │
   │ Codestral 22B (MNPL)     │               │ embeddings, moderation   │
   └───────────┬──────────────┘               └────────────┬─────────────┘
               │  transformers / vLLM / Ollama              │  `mistralai` SDK
               ▼                                            ▼  or raw HTTPS
        your GPU/CPU                                  pip install mistralai
               └───────────────────┬────────────────────────┘
                                   ▼
            same chat format · same tool-calling shape · same lab
```

Three things to internalize:

1. **Two deployment modes, one mental model.** Open weights = you are the inference provider (pick `transformers`/vLLM/Ollama). API = they host it (`mistralai` SDK, OpenAI-compatible JSON). The *prompt format and tool-calling shape are shared*, so moving between them is cheap.
2. **MoE is the efficiency trick.** Mixtral routes each token to **2 of 8 expert FFNs**. Total weights are huge (must fit in memory), but only a fraction *compute* per token — so inference speed/cost tracks the **active** parameters, not the total.
3. **The instruct format is `[INST]…[/INST]`, not ChatML.** Mistral's own template wraps user turns in `[INST] … [/INST]` with a leading `<s>` BOS. Get it wrong and quality drops — use the tokenizer's `apply_chat_template` (or `mistral-common`) rather than hand-rolling.

## 3. Key Concepts

- **Open-weight vs premier (closed).** Mistral 7B / Mixtral / Nemo / Small are **open-weight, Apache 2.0**. **Mistral Large, Pixtral Large** are **closed, API-only**. **Codestral** sits between (weights under the non-production **MNPL** license; commercial use via API). Always check the license per model.
- **Mixture-of-Experts (MoE).** Mixtral replaces each dense FFN with **8 experts** + a router that picks the **top-2** per token. **Total** params (must be in memory) ≫ **active** params (drive compute). Mixtral 8x7B ≈ 46.7B total / ~12.9B active.
- **Sliding-Window Attention (SWA).** Mistral 7B attends only to the last *W* tokens (W=4096) per layer; stacking layers gives a large effective receptive field at linear cost. Cheap long-context attention.
- **Grouped-Query Attention (GQA).** Multiple query heads share fewer key/value heads — shrinks the KV-cache and speeds inference vs full multi-head attention.
- **`[INST]` chat template.** Mistral instruct format: `<s>[INST] {user} [/INST] {assistant}</s>[INST] {next user} [/INST]`. System content is folded into the first `[INST]`. Use `apply_chat_template`, don't hand-concatenate.
- **Tokenizers (`mistral-common` / tekken).** Newer models (Nemo, Large 2) use the **tekken** tokenizer (tiktoken-based, ~130K vocab); older ones use SentencePiece. The `mistral-common` library is the reference tokenizer and request validator.
- **Function / tool calling.** La Plateforme supports OpenAI-style `tools` + `tool_choice`; the model returns `tool_calls` you execute and feed back. Open weights support the same via their chat template's `[TOOL_CALLS]` tokens.
- **Fill-in-the-Middle (FIM).** Codestral's headline feature: given a prefix and suffix it completes the **middle** — the primitive behind IDE autocomplete. Exposed via a dedicated `/fim/completions` endpoint.
- **La Plateforme & `mistralai` SDK.** The hosted API (chat, embeddings, FIM, vision, moderation, fine-tuning). Python SDK `pip install mistralai`; also reachable as raw HTTPS and via an OpenAI-compatible base URL.
- **VRAM rule of thumb.** Memory ≈ `total_params × bytes/weight` (MoE counts **total**, not active) + KV-cache. Mixtral 8x7B at 4-bit ≈ 24 GB+; Mistral 7B at 4-bit fits in ~6 GB.

## 4. Setup

Three common paths. **API** is fastest to *try*; **Ollama** is easiest to *self-host*; **transformers** is the path for research/fine-tuning.

```bash
# Path A — Hosted API (La Plateforme). Get a key at https://console.mistral.ai
pip install mistralai
export MISTRAL_API_KEY=...        # then call mistral-small-latest, mistral-large-latest, codestral-latest, ...

# Path B — Ollama (easiest local; bundles a quantized GGUF)
#   install from https://ollama.com, then:
ollama run mistral               # Mistral 7B, CPU-OK
ollama run mixtral               # Mixtral 8x7B MoE (needs more RAM/VRAM)

# Path C — Hugging Face transformers (research / fine-tuning)
pip install "transformers>=4.43" torch accelerate
#   open repos (Apache-2.0, no token needed): mistralai/Mistral-7B-Instruct-v0.3
```

A minimal API call looks like this (gated below behind a key check so the notebook always runs):

```python
from mistralai import Mistral
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
resp = client.chat.complete(
    model="mistral-small-latest",
    messages=[{"role": "user", "content": "Hello"}],
)
print(resp.choices[0].message.content)
```

The cells below run top-to-bottom in a fresh kernel **without** the SDK, a key, `torch`, or any weights — every network/inference call is gated behind an `os.getenv` check, while the no-network examples always execute.

In [ ]:
# This notebook executes with or without the SDK, an API key, or model weights.
# To run the live API example:  pip install mistralai  and  export MISTRAL_API_KEY=...
import os

api_key = os.getenv("MISTRAL_API_KEY")

try:
    import mistralai  # noqa: F401
    have_sdk = True
except ImportError:
    have_sdk = False

print("MISTRAL_API_KEY :", "set" if api_key else "(unset — live API calls skipped)")
print("mistralai SDK   :", "installed" if have_sdk else "(not installed)")
print("\nOpen weights are Apache-2.0 (Mistral 7B, Mixtral, Nemo, Small);")
print("Mistral Large / Pixtral are closed (API only); Codestral is MNPL.")

## 5. Worked Examples

### Example 1 — MoE arithmetic: total vs *active* parameters (no network)

The single most important Mistral idea is **mixture-of-experts**: Mixtral has a huge parameter *count* (so it needs a lot of memory) but only *activates* a couple of experts per token (so it computes — and costs — like a much smaller model). The estimator below makes that split concrete.

In [ ]:
# Why MoE is efficient: memory tracks TOTAL params, speed/cost track ACTIVE params.
def moe_params(experts, top_k, expert_b, shared_b):
    """Return (total_B, active_B) in billions for a top-k MoE.

    expert_b = params per single expert (billions); shared_b = everything that's
    NOT an expert (attention, embeddings, router, norms) and is always active.
    """
    total  = shared_b + experts * expert_b           # all experts live in memory
    active = shared_b + top_k   * expert_b            # only top_k run per token
    return total, active

# Approximate decomposition of Mixtral 8x7B (8 experts, top-2 routed).
total, active = moe_params(experts=8, top_k=2, expert_b=5.6, shared_b=1.9)
print(f"Mixtral 8x7B : {total:5.1f}B total  ->  {active:5.1f}B active per token")

# A dense 7B for comparison: total == active (every weight runs every token).
print(f"Mistral 7B   :   7.2B total  ->    7.2B active per token (dense)")

ratio = total / active
print(f"\nMixtral holds {ratio:.1f}x more weights than it runs per token:")
print(f"  -> needs memory for ~{total:.0f}B params (the gotcha)")
print(f"  -> but inference compute/latency is ~{active:.0f}B-class (the win)")

### Example 2 — Build Mistral's `[INST]` chat prompt by hand (no network)

Mistral instruct models were trained on a precise token stream — **not** the ChatML / `<|im_start|>` format other models use. The wrapper is `<s>[INST] … [/INST] …</s>`. Reproducing it by hand puts the format in muscle memory and shows where the system prompt and tool results go.

In [ ]:
# The Mistral v3 instruct chat format — pure string assembly, no model needed.
# This mirrors what tokenizer.apply_chat_template(...) emits.
BOS, EOS = "<s>", "</s>"

def mistral_prompt(messages):
    """Reproduce Mistral's [INST] template (system folded into first user turn)."""
    sys = next((m["content"] for m in messages if m["role"] == "system"), None)
    turns = [m for m in messages if m["role"] != "system"]
    out, first = [BOS], True
    for m in turns:
        if m["role"] == "user":
            content = m["content"]
            if first and sys:                       # system prompt prepended once
                content = f"{sys}\n\n{content}"
            out.append(f"[INST] {content} [/INST]")
            first = False
        else:                                       # assistant turn closes with EOS
            out.append(f" {m['content']}{EOS}")
    return "".join(out)

messages = [
    {"role": "system",    "content": "You are a terse assistant."},
    {"role": "user",      "content": "Capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user",      "content": "And of Italy?"},
]
print(mistral_prompt(messages))
print("\n--- the model generates after the final [/INST], stopping at </s> ---")

### Example 3 — Call La Plateforme (gated on `MISTRAL_API_KEY`)

The real hosted path. With `MISTRAL_API_KEY` set and `pip install mistralai`, this sends a live chat request; otherwise it prints the exact call shape so the notebook still executes cleanly. We use raw `urllib` for the no-SDK fallback shape and the SDK when available — the JSON body is OpenAI-compatible.

In [ ]:
# Live API call, gated so the notebook runs with or without a key / the SDK.
import os, json, urllib.request

def mistral_chat_sdk(prompt, model="mistral-small-latest"):
    from mistralai import Mistral                      # pip install mistralai
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    resp = client.chat.complete(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

def mistral_chat_http(prompt, model="mistral-small-latest"):
    """Same request without the SDK — the body is OpenAI-compatible."""
    body = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
    }).encode()
    req = urllib.request.Request(
        "https://api.mistral.ai/v1/chat/completions", data=body,
        headers={"Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}",
                 "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)["choices"][0]["message"]["content"]

if api_key:
    try:
        fn = mistral_chat_sdk if have_sdk else mistral_chat_http
        print("mistral says:", fn("Reply with exactly one word: pong").strip())
    except Exception as e:                              # network / auth / quota
        print("Live call failed:", type(e).__name__, e)
else:
    print("Skipping live API call (set MISTRAL_API_KEY to run).")
    print("Call shape (SDK):")
    print("  client = Mistral(api_key=...)")
    print("  client.chat.complete(model='mistral-small-latest', messages=[...])")
    print("Endpoint: POST https://api.mistral.ai/v1/chat/completions  (OpenAI-compatible)")
    print("Code FIM: POST https://api.mistral.ai/v1/fim/completions  (Codestral prefix+suffix)")

### Example 4 — Self-host via Ollama (call shape, gated on `RUN_OLLAMA`)

For "just run Mistral on my machine," **Ollama** is usually the answer: one binary, automatic GGUF quantization, an OpenAI-compatible HTTP endpoint. The shape below is what you type and call; it's gated since it needs the Ollama daemon running with the model pulled.

In [ ]:
# Local self-hosting via Ollama, gated since it needs the daemon + a pulled model.
import os, json, urllib.request

OLLAMA_URL = "http://localhost:11434/api/chat"

def ollama_chat(prompt, model="mistral"):
    body = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }).encode()
    req = urllib.request.Request(OLLAMA_URL, data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)["message"]["content"]

if os.getenv("RUN_OLLAMA") == "1":
    try:
        print("ollama says:", ollama_chat("Reply with one word: pong").strip())
    except Exception as e:
        print("Ollama call failed:", type(e).__name__, e)
else:
    print("Skipping Ollama call (set RUN_OLLAMA=1 with the daemon running).")
    print("Setup:  ollama run mistral        # Mistral 7B, Apache-2.0, CPU-OK")
    print("        ollama run mixtral        # Mixtral 8x7B MoE (more RAM/VRAM)")
    print("HTTP :  POST http://localhost:11434/api/chat  {model, messages, stream}")
    print("transformers path: mistralai/Mistral-7B-Instruct-v0.3 (no HF token needed)")

## 6. Gotchas & Pitfalls

- **Assuming every Mistral model is Apache 2.0.** The open line (7B, Mixtral, Nemo, Small) is — but **Mistral Large and Pixtral are closed (API only)**, and **Codestral's weights are MNPL** (non-production). Check the license per model before you self-host commercially.
- **MoE memory shock.** Mixtral *computes* like ~13B but you must hold **all** ~47B params in memory. People size hardware off the active count and OOM. Memory tracks **total**; speed tracks **active**.
- **Wrong chat template.** Mistral uses `<s>[INST] … [/INST]`, **not** ChatML (`<|im_start|>`) or Llama 3's `<|start_header_id|>`. Hand-rolling the wrong format silently degrades output. Use `apply_chat_template` / `mistral-common`.
- **System-prompt handling.** Mistral has no dedicated system token in the classic template — system content is **folded into the first `[INST]`**. Sticking a separate system turn in the raw string can confuse older models.
- **Tokenizer mismatch.** Newer models (Nemo, Large 2) use the **tekken** tokenizer; older ones use SentencePiece. Mixing the wrong tokenizer with a model yields garbage and wrong token counts/costs.
- **Using a base model for chat.** The base checkpoints aren't instruction-tuned — load the `-Instruct` repo for assistant behavior.
- **Codestral ≠ chat model.** Codestral shines at code completion / **FIM** via the dedicated endpoint; using it like a general chat model (or vice-versa using a chat model for FIM) leaves quality on the table.
- **`-latest` aliases move.** `mistral-large-latest` silently repoints to new versions. Pin a dated version (e.g. `mistral-large-2411`) when you need reproducibility.
- **Expecting frontier reasoning everywhere.** Mistral's small open models are efficient, not magic — for the hardest reasoning/coding, Mistral Large or a top closed model will outperform the 7B/12B. Match model size to task difficulty.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Open weights with a *truly* permissive license** | **Mistral 7B / Mixtral / Nemo / Small** (Apache 2.0) | No usage-cap clause — unlike Llama's Community License. Self-host, fine-tune, ship commercially. |
| **Big-model quality at small-model inference cost** | **Mixtral (MoE)** | Activates ~13B of ~47B per token; quality-per-FLOP is the whole point. |
| **One vendor, prototype-on-API then self-host** | **Mistral** | Same lab, shared prompt format across the open + hosted models. |
| **Code autocomplete / fill-in-the-middle** | **Codestral** | Purpose-built FIM endpoint for IDE-style completion. |
| **EU data residency** | **Mistral** | EU-headquartered with EU hosting options. |
| **Absolute frontier reasoning/coding, zero ops** | **Claude / GPT / Gemini** | Top closed models still edge out Mistral Large on the hardest tasks, fully hosted. |
| **Other open-weight families** | **Llama / Qwen / DeepSeek / Gemma** | Qwen leads multilingual/coding, Llama has the biggest ecosystem, DeepSeek leads reasoning-per-dollar. |

**Honest trade-offs:**

- **vs closed frontier APIs (Claude, GPT, Gemini)** — Mistral trades a bit of peak quality for **open weights, a permissive license, efficiency, and EU residency**. If you don't need those, a hosted frontier API is often smarter with less effort. A common pattern: open Mistral for bulk traffic, a frontier API for the hard 5%.
- **vs Llama** — both are open-weight families; the key differentiators are **license** (Mistral's Apache 2.0 has no MAU clause vs Llama's Community License) and **MoE efficiency** (Mixtral). Llama has the larger ecosystem and tooling. Benchmark on *your* task.
- **vs Qwen / DeepSeek / Gemma** — Qwen often leads coding/multilingual; DeepSeek is strong on reasoning value; Gemma is Google's small-model line. Mistral's edge is efficiency + the clean Apache license + the integrated API.
- **vs running Mistral through a hosted provider** (Together, Fireworks, AWS Bedrock, Azure) — rent Mistral inference instead of self-hosting: open-model flexibility without owning GPUs, at a per-token price. A sensible middle ground.

**Rule of thumb:** reach for Mistral when you want **open weights with a clean license and strong efficiency** — and optionally a same-lab API to burst to. Default to a closed frontier model when you just want the smartest possible answer with no infrastructure, and to Llama/Qwen when their ecosystem or task-specific scores fit better.

## 8. Resources

- **Mistral docs (La Plateforme)** — https://docs.mistral.ai/
- **Models overview & licensing** — https://docs.mistral.ai/getting-started/models/models_overview/
- **Python SDK (`mistralai`)** — https://github.com/mistralai/client-python
- **`mistral-common` (reference tokenizer & chat templates)** — https://github.com/mistralai/mistral-common
- **Mistral 7B paper (sliding-window + GQA)** — https://arxiv.org/abs/2310.06825
- **Mixtral of Experts paper (MoE)** — https://arxiv.org/abs/2401.04088
- **Mistral models on Hugging Face** — https://huggingface.co/mistralai
- **Ollama (easiest local inference)** — https://ollama.com/library/mistral

**Related notebooks:** `llama`, `qwen`, `deepseek` (other open-weight families for head-to-head); `anthropic-claude-api`, `google-gemini` for the open-vs-closed / self-host-vs-hosted trade-off.